# Rule-Consistency Auditor
- Francesco Buda, francesco.buda3@studio.unibo.it
- Emanuele Sanchi, emanuele.sanchi@studio.unibo.it
- Tommaso Severi, tommaso.severi2@studio.unibo.it

## Import libraries

In [62]:
import time
import carla_utils
import data_list_bind
import carla
import pygame
import datetime
import json
import threading

## Setup log file and CARLA world

In [63]:
log_filename = f"logs/rca_log_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
world, spectator, client = carla_utils.world_connect()

## Spawn vehicles

In [64]:
AUTOPILOT_VEHICLE_COUNT = 10
for i in range(AUTOPILOT_VEHICLE_COUNT):
    carla_utils.spawn_random_vehicle_no_bike(world, spawn_index=i, autopilot=True)
# vehicle_autopilot = carla_utils.spawn_random_vehicle_no_bike(world, spawn_index=0, autopilot=True)
ego_vehicle = carla_utils.spawn_vehicle(world, spawn_index=10, autopilot=False)

## Setup binder thread

In [65]:
binder = data_list_bind.VariableBinder(world, ego_vehicle)
scene_data_shared = {}
scene_data_lock = threading.Lock()
binder_stop_event = threading.Event()

def binder_thread_fn():
    while not binder_stop_event.is_set():
        data = binder.compute_scene_data()
        with scene_data_lock:
            scene_data_shared.update(data)
        time.sleep(0.05)

binder_thread = threading.Thread(target=binder_thread_fn, daemon=True)
binder_thread.start()

In [66]:
'''pygame.init()
pygame.joystick.init()
screen = pygame.display.set_mode((400, 100))
pygame.display.set_caption("Joystick inspector — Q to quit")

joysticks = [pygame.joystick.Joystick(i) for i in range(pygame.joystick.get_count())]
for j in joysticks:
    j.init()
    print(f"Joystick found: {j.get_name()} | axes={j.get_numaxes()} buttons={j.get_numbuttons()} hats={j.get_numhats()}")

running = True
while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_q:
                running = False
        elif event.type == pygame.JOYAXISMOTION:
            if abs(event.value) > 0.05:  # filtra rumore
                print(f"AXIS     axis={event.axis}  value={event.value:.3f}")
        elif event.type == pygame.JOYBUTTONDOWN:
            print(f"BUTTON   button={event.button}")
        elif event.type == pygame.JOYHATMOTION:
            print(f"HAT      hat={event.hat}  value={event.value}")

pygame.quit()'''

'pygame.init()\npygame.joystick.init()\nscreen = pygame.display.set_mode((400, 100))\npygame.display.set_caption("Joystick inspector — Q to quit")\n\njoysticks = [pygame.joystick.Joystick(i) for i in range(pygame.joystick.get_count())]\nfor j in joysticks:\n    j.init()\n    print(f"Joystick found: {j.get_name()} | axes={j.get_numaxes()} buttons={j.get_numbuttons()} hats={j.get_numhats()}")\n\nrunning = True\nwhile running:\n    for event in pygame.event.get():\n        if event.type == pygame.QUIT:\n            running = False\n        elif event.type == pygame.KEYDOWN:\n            if event.key == pygame.K_q:\n                running = False\n        elif event.type == pygame.JOYAXISMOTION:\n            if abs(event.value) > 0.05:  # filtra rumore\n                print(f"AXIS     axis={event.axis}  value={event.value:.3f}")\n        elif event.type == pygame.JOYBUTTONDOWN:\n            print(f"BUTTON   button={event.button}")\n        elif event.type == pygame.JOYHATMOTION:\n       

## Main control loop

In [67]:
from controller.state_machine import StateMachine
import parsing_utils
from models.state import StateType

state_machines = [StateMachine(rule) for rule in parsing_utils.parse_rule_files("admin/rules/")]

In [68]:
pygame.init()
screen = pygame.display.set_mode((300, 80))
pygame.display.set_caption("RCA Control  |  WASD · R=reverse · Q=quit")

control        = carla.VehicleControl()
throttle_step  = 0.03
steer_step     = 0.04
brake_step     = 0.1
running        = True
frame_count    = 0
reverse        = False
log_buffer     = []
FLUSH_EVERY    = 20
current_lights = int(carla.VehicleLightState.NONE)

tick_event = threading.Event()

def on_server_tick(snapshot):
    tick_event.set()

world.on_tick(on_server_tick)

def clamp(v, lo, hi):
    return max(lo, min(hi, v))

joystick = pygame.joystick.Joystick(0) if pygame.joystick.get_count() > 0 else None
if joystick:
    joystick.init()

try:
    while running:
        tick_event.wait(timeout=0.1)
        tick_event.clear()

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            elif event.type == pygame.KEYDOWN:
                if event.key in (pygame.K_q, pygame.K_ESCAPE):
                    running = False
                elif event.key == pygame.K_r:
                    reverse = not reverse
            elif event.type == pygame.JOYBUTTONDOWN:
                if event.button == 0:
                    reverse = not reverse
                    if reverse:
                        current_lights |= int(carla.VehicleLightState.Reverse)
                    else:
                        current_lights &= ~int(carla.VehicleLightState.Reverse)
                    ego_vehicle.set_light_state(carla.VehicleLightState(current_lights))
                elif event.button == 1:
                    current_lights ^= int(carla.VehicleLightState.LeftBlinker)
                    current_lights ^= int(carla.VehicleLightState.RightBlinker)
                    ego_vehicle.set_light_state(carla.VehicleLightState(current_lights))
                elif event.button == 4:
                    current_lights ^= int(carla.VehicleLightState.LeftBlinker)
                    ego_vehicle.set_light_state(carla.VehicleLightState(current_lights))
                elif event.button == 5:
                    current_lights ^= int(carla.VehicleLightState.RightBlinker)
                    ego_vehicle.set_light_state(carla.VehicleLightState(current_lights))

        keys = pygame.key.get_pressed()

        if joystick:
            steer_raw = joystick.get_axis(0)
            if abs(steer_raw) > 0.05:
                control.steer = steer_raw
            else:
                control.steer -= control.steer * 0.1

            accel_raw = joystick.get_axis(1)
            throttle_joy = (accel_raw + 1.0) / 2.0

            brake_raw = joystick.get_axis(2)
            brake_joy = (brake_raw + 1.0) / 2.0

            if throttle_joy > 0.05:
                control.throttle = throttle_joy
                control.brake = 0.0
            elif brake_joy > 0.05:
                control.brake = brake_joy
                control.throttle = 0.0
            else:
                control.throttle = 0.0
                control.brake = 0.0
        else:
            if keys[pygame.K_w]:
                control.throttle = clamp(control.throttle + throttle_step, 0.0, 1.0)
                control.brake = 0.0
            elif keys[pygame.K_s]:
                control.brake = clamp(control.brake + brake_step, 0.0, 1.0)
                control.throttle = 0.0
            else:
                control.throttle = 0.0
                control.brake = 0.0

            if keys[pygame.K_a]:
                control.steer = clamp(control.steer - steer_step, -1.0, 1.0)
            elif keys[pygame.K_d]:
                control.steer = clamp(control.steer + steer_step, -1.0, 1.0)
            else:
                control.steer -= control.steer * 0.1

        control.reverse = reverse
        ego_vehicle.apply_control(control)

        carla_utils.move_spectator_to(
            ego_vehicle.get_transform(), spectator,
            distance=4.0, z=2.0, pitch=-10
        )

        with scene_data_lock:
            scene_data = dict(scene_data_shared)
        timestamp = datetime.datetime.now().strftime('%H:%M:%S.%f')[:-3]

        log_entry = {
            'frame':     frame_count,
            'timestamp': timestamp,
            'reverse':   reverse,
            'control': {
                'throttle': round(float(control.throttle), 3),
                'brake':    round(float(control.brake),    3),
                'steer':    round(float(control.steer),    3),
            },
            'scene_data': {
                k: str(v) if hasattr(v, '__dict__') else v
                for k, v in scene_data.items()
            }
        }
        log_buffer.append(log_entry)

        if len(log_buffer) >= FLUSH_EVERY:
            with open(log_filename, 'a') as f:
                for entry in log_buffer:
                    f.write(json.dumps(entry) + '\n')
            log_buffer.clear()

        frame_count += 1
        if frame_count % 10 == 0:
            for sm in state_machines:
                current_state = sm.evaluate(scene_data)
                # print(f"Evaluated {sm._rule.name}: current state = {current_state}")
                if current_state.type == StateType.VIOLATION:
                    print(f"Rule violation detected: {sm._rule.name} at frame {frame_count}")

finally:
    binder_stop_event.set()
    binder_thread.join(timeout=2.0)

    if log_buffer:
        with open(log_filename, 'a') as f:
            for entry in log_buffer:
                f.write(json.dumps(entry) + '\n')

    try:
        ego_vehicle.destroy()
    except Exception:
        pass
    try:
        for actor in world.get_actors().filter('vehicle*'):
            if actor.id != ego_vehicle.id:
                actor.destroy()
    except Exception:
        pass

    pygame.quit()
    print(f"Simulazione terminata dopo {frame_count} frame. Log: {log_filename}")

Rule violation detected: lane_keeping at frame 370
Rule violation detected: lane_keeping at frame 380
Rule violation detected: lane_keeping at frame 390
Rule violation detected: lane_keeping at frame 400
Rule violation detected: lane_keeping at frame 410
Rule violation detected: lane_keeping at frame 530
Rule violation detected: lane_keeping at frame 540
Rule violation detected: lane_keeping at frame 690
Rule violation detected: lane_keeping at frame 700
Rule violation detected: lane_keeping at frame 710
Rule violation detected: lane_keeping at frame 720
Rule violation detected: lane_keeping at frame 730
Rule violation detected: lane_keeping at frame 740
Rule violation detected: lane_keeping at frame 750
Rule violation detected: lane_keeping at frame 790
Rule violation detected: lane_keeping at frame 830
Rule violation detected: lane_keeping at frame 840
Rule violation detected: lane_keeping at frame 850
Rule violation detected: lane_keeping at frame 860
Rule violation detected: lane_k

## Cleanup

In [69]:
print("Destroying all vehicles in the simulator...")
for actor in world.get_actors().filter('vehicle.*'):
    try:
        actor.destroy()
        print(f"  Destroyed: {actor.id}")
    except Exception as e:
        print(f"  Skip {actor.id}: {e}")
print("All vehicles cleaned up. Ready for next run!")

Destroying all vehicles in the simulator...
All vehicles cleaned up. Ready for next run!
